In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Embedding
from keras.utils import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [23]:
spam = pd.read_csv('spam.csv')
spam.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [24]:
labelencoder = LabelEncoder()
y = labelencoder.fit_transform(spam['Category'])

In [25]:
messages = spam['Message'].values
X_train, X_test, y_train, y_test = train_test_split(messages, y, test_size=0.3)

In [26]:
token = Tokenizer(num_words=1000)
token.fit_on_texts(X_train)
X_train = token.texts_to_sequences(X_train)
X_test = token.texts_to_sequences(X_test)

In [27]:
print(X_train)

[[160, 291, 34, 26, 169, 4, 422, 483, 259, 119, 103, 62, 211], [865, 134, 117, 372, 45, 1, 57, 3, 525], [34, 16, 49, 23, 25], [90, 186, 2, 353, 501, 79, 24, 87, 423, 74], [117, 1, 57, 25, 13, 484, 114], [16, 4, 59, 799, 396, 51, 12, 373, 52, 3, 33, 91, 34, 397, 80], [177, 4, 170, 39, 41, 6, 27, 4, 116, 21, 800, 18, 47, 109, 19, 147, 72, 109, 19, 27, 147, 2, 398, 72], [6, 801, 10, 302, 245, 174, 125, 1, 19, 439, 6, 24, 1, 292, 943, 45, 6, 802, 80, 10, 52, 47, 1, 6], [31, 944, 2, 14, 1, 38, 129, 11, 98, 599, 10, 526, 217, 67, 4, 329, 303, 40, 293, 40, 803, 10, 9, 184, 9, 56, 115], [1, 57, 9, 13, 374, 804], [21, 3, 87, 39], [1, 38, 485, 4, 738, 23, 161], [126, 21, 3, 117], [13, 486, 56, 2, 46, 200, 2, 39, 10, 375, 4, 178, 9, 5, 424, 23, 1, 16, 2, 294, 39, 13, 353, 70], [304, 425, 47, 8, 4, 422, 682, 92, 14, 8, 118, 641, 20, 4, 52, 14, 8, 283, 439, 82, 4, 59, 123], [31, 28, 151, 22, 24, 31, 9, 11, 330, 23, 1, 207, 866, 331, 555], [3, 60, 265, 9, 3, 36, 33, 190, 218, 9, 181, 9, 19, 27, 147,

In [28]:
X_train = pad_sequences(X_train, padding='post', maxlen=500)
X_test = pad_sequences(X_test, padding='post', maxlen=500)

In [29]:
X_train

array([[160, 291,  34, ...,   0,   0,   0],
       [865, 134, 117, ...,   0,   0,   0],
       [ 34,  16,  49, ...,   0,   0,   0],
       ...,
       [284, 318,   6, ...,   0,   0,   0],
       [ 89, 220,   1, ...,   0,   0,   0],
       [ 58,   4,  95, ...,   0,   0,   0]], dtype=int32)

In [30]:
model = Sequential()
model.add(Embedding(input_dim=len(token.word_index), output_dim=50, input_length=500))
model.add(Flatten())
model.add(Dense(units=10, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(units=1, activation='sigmoid'))

/Users/caio.motta/.pyenv/versions/3.11.11/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [31]:
model.compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
model.fit(X_train, y_train, epochs=20, batch_size=10, verbose=True, validation_data=(X_test, y_test))

Epoch 1/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8610 - loss: 0.1203 - val_accuracy: 0.9474 - val_loss: 0.0538
Epoch 2/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9564 - loss: 0.0490 - val_accuracy: 0.9761 - val_loss: 0.0406
Epoch 3/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9749 - loss: 0.0383 - val_accuracy: 0.9797 - val_loss: 0.0336
Epoch 4/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9802 - loss: 0.0303 - val_accuracy: 0.9797 - val_loss: 0.0291
Epoch 5/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9785 - loss: 0.0268 - val_accuracy: 0.9809 - val_loss: 0.0254
Epoch 6/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9842 - loss: 0.0216 - val_accuracy: 0.9755 - val_loss: 0.0259
Epoch 7/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9842 - loss: 0.0199 - val_accuracy: 0.9827 - val_loss: 0.0220
Epoch 8/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9826 - loss: 0.0189 - val_accuracy: 0.

In [34]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Loss: {loss}')
print(f'Accuracy: {accuracy}')

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9790 - loss: 0.0167
Loss: 0.017120013013482094
Accuracy: 0.9784688949584961


In [35]:
new_prediction = model.predict(X_test)
print(new_prediction)

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
[[9.9984497e-01]
 [3.1643478e-15]
 [2.7121623e-06]
 ...
 [7.0057897e-27]
 [2.8906278e-15]
 [2.7194150e-07]]


In [36]:
pred = (new_prediction > 0.5)
print(pred)

[[ True]
 [False]
 [False]
 ...
 [False]
 [False]
 [False]]


In [37]:
cm = confusion_matrix(y_test, pred)
print(cm)

[[1429    5]
 [  31  207]]
